###### Spark Session:

In [0]:
## Getting the details of already existing spark session.
print(spark)

In [0]:
## In case, we want to create new spark session, we can go with

## Step1: Import SparkSession from pyspark.sql
from pyspark.sql import SparkSession

## Step2: Create a sparksession called SparkLearn
spark = SparkSession.builder.appName("SparkLearn").getOrCreate()

## Note: We can create a spark session on local machine, but on databricks, it is managed by databricks, so no need to create it explicitly.

###### Create Dataframe with Data:

In [0]:
## import the required libraries, specially 2nd and 3rd are used below for StructType and StructField
from pyspark.sql import SparkSession
## We want to import functions as, it have col(), StructType and StructField function
from pyspark.sql.functions import *
## We want to import types as, it have StringType(), IntegerType() and all other in it.
from pyspark.sql.types import *

In [0]:
## declare the data and the schema, and then create a dataframe from it
data =[
    ("Sushant",25,"New York"),
    ("Anjali",22,"San Francisco"),
    ("Ravi",27,"Chicago"),
    ("Priya",24,"Seattle")
]

schema = StructType([
    StructField("Name",StringType(),True),
    StructField("Age",IntegerType(),True),
    StructField("City",StringType(),True)
])

df = spark.createDataFrame(data,schema)

###### Transformation: Filter:

In [0]:
## Filter the data for City = New York
df = df.filter(col("city")=='New York')

## Note: It is a example of "Lazy Evaluation", where the data is not actually filtered, but the filter is applied on the dataframe, and the data is filtered only when it is needed and if any "action" is performed.

In [0]:
## The "Action"
display(df)

###### Transformation: Filter & Select:

In [0]:
## declare the data and the schema, and then create a dataframe from it
data =[
    ("Sushant",25,"New York"),
    ("Anjali",22,"San Francisco"),
    ("Ravi",27,"Chicago"),
    ("Priya",24,"Seattle")
]

schema = StructType([
    StructField("Name",StringType(),True),
    StructField("Age",IntegerType(),True),
    StructField("City",StringType(),True)
])

new_df = spark.createDataFrame(data,schema)

In [0]:
df_new = new_df.filter(col('Name')=='Sushant')

In [0]:
df_new = df_new.select('City')

In [0]:
display(df_new)

In [0]:
df_new.explain()
## Note: The explain() function is not supported in serverless mode, consider using df_new.explain() only in dedicated mode or alternative methods.

###### Repartition VS Coalesce:

In [0]:
df_new.rdd.getNumPartitions()  
# Not supported in serverless mode, consider using df_new.rdd.getNumPartitions() only in dedicated mode or alternative methods.

In [0]:
## If you want to do the repartition, you can do it as below
df_new = df_new.repartition(3)
df_new.rdd.getNumPartitions()

In [0]:
## If we want to again coalesce it to 1 partition, we can do it as below
df_new = df_new.coalesce(1)
df_new.rdd.getNumPartitions()

###### Jobs, Stages & Tasks:

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
## Action 1 - .read()
## Action 2 - inferschema - Because inferSchema sees the data and decides the best schema
df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("/Workspace/Users/official.sushantshanu@gmail.com/Drafts/IPL-Analysis/Raw_Data/Team.csv")

df.display()

In [0]:
## Narrow Transformation 1:
df = df.filter(col("Team_Id")>5)

In [0]:
## Narrow Transformation 2:
df = df.select('Team_Id','Team_Name')

In [0]:
## Wide Transformation:
df = df.groupBy('Team_Name').agg(count(col('Team_Id')))

In [0]:
df.display()

###### Joins:

In [0]:
## Used to disable broadcast join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",-1)


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast

data1 = [
    (1,"Alice"),
    (2,"Bob"),
    (3,"Sushant"),
    (4,"Anjali"),
    (5,"Ravi")
]
df1 = spark.createDataFrame(data1,["id","name"])

data2 = [
    (1,1000),
    (2,2000),
    (3,5000),
    (4,8000)
]
df2 = spark.createDataFrame(data2,["id","salary"])

In [0]:
df_join = df1.join(df2,df1['id']==df2['id'],'left')

In [0]:
df_join.display()

In [0]:
## To perform broadcast join, apply broadcast on smaller table:
df_join_broadcast = df1.join(broadcast(df2),df1['id']==df2['id'],'left')

In [0]:
df_join_broadcast.display()

###### Caching:

In [0]:
from pyspark.sql.functions import *

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import SparkSession

data1 = [
    (1,"Alice"),
    (2,"Bob"),
    (3,"Sushant"),
    (4,"Anjali"),
    (5,"Ravi")
]
df1 = spark.createDataFrame(data1,["id","name"])

In [0]:
df1 = df1.withColumn('ActiveFlag',lit('Yes'))

In [0]:
display(df1)

In [0]:
df1.cache()

In [0]:
df2 = df1.filter(col('name')=="Sushant")

In [0]:
display(df2)

In [0]:
df2.explain()

###### Persist:

In [0]:
from pyspark.storagelevel import StorageLevel

In [0]:
df1.persist(StorageLevel.MEMORY_ONLY)
# or
# df1.persist(StorageLevel.MEMORY_AND_DISK)
# To unpersist: df1.unpersist()

###### Partition Pruning:

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import SparkSession

data1 = [
    (1,"Alice","HR"),
    (2,"Bob","HR"),
    (3,"Sushant","IT"),
    (4,"Anjali","IT"),
    (5,"Ravi","Fin")
]
df1 = spark.createDataFrame(data1,["id","name","department"])

In [0]:
output_path = "/Workspace/Users/official.sushantshanu@gmail.com/Drafts/Output_data"

In [0]:
df1.write\
    .mode("overwrite")\
    .partitionBy("department")\
    .parquet(output_path)


In [0]:
# Check the status of Adaptive Query Execution (AQE)
spark.conf.get("spark.sql.adaptive.enabled")

# Enable & Disable Adaptive Query Execution (AQE)
spark.conf.set("spark.sql.adaptive.enabled",False) # Disable
spark.conf.set("spark.sql.adaptive.enabled",True) # Enable